## **Feature Engineering** ##

Feature engineering is the process of adding new variables, which may affect the value of the target variable, to the table. 

In this project, covid-19 period, SARS period, population, and school days (weeks) might affect the number of influenza detections, according to our prior knowledge. 

---

In [1]:
# Import libraries needed
import os

import pandas as pd
import numpy as np

In [2]:
# Import data and settings
df = pd.read_csv("data.csv")

df["From"] = pd.to_datetime(df["From"])


---
### **1. Handling for COVID-19 period**
`pandemic_flag`: a Binary flay used to indicate whether the specific week is in COVID-19 period or not

`ContainmentHealthIndex_Average`: an index indicate how pandemic affects influenza transmission specifically in Hong Kong
- obtained from OxCGRT Github, done by the University of Oxford Blavatink School of Government  https://www.bsg.ox.ac.uk/research/covid-19-government-response-tracker
- answers "How" instead of Tes or No
- a scale from 0 to 100 

In [3]:
# Retreive the OxCGRT index and put it inside a DataFrame along with date
path = os.getcwd()
path += "\\dataset\\OxCGRT_compact_national_v1.csv"
df_oxcgrt = pd.read_csv(path)
df_oxcgrt = df_oxcgrt[df_oxcgrt["CountryCode"] == "HKG"]
df_oxcgrt["Date"] = pd.to_datetime(df_oxcgrt["Date"], format="%Y%m%d")

df_health = df_oxcgrt[["Date", "ContainmentHealthIndex_Average"]].copy()

# Add a new column <ContainmentHealthIndex_Average> to df to control data in the covid period (add weights to them)
df = pd.merge(df, df_health, left_on="From", right_on="Date", how="left")
df["ContainmentHealthIndex_Average"] = df["ContainmentHealthIndex_Average"].fillna(0.0)

# Add a new column <pandemic_flag> to df to classify covid period or not
covid_index = df[((df["From"] >= "2020-01-03") & (df["From"] <= "2023-03-01"))].index
df.loc[covid_index, "pandemic_flag"] = 1
#df["pandemic_flag"].fillna(0, inplace=True)
df.fillna({"pandemic_flag": 0}, inplace=True)

# Drop unnecessary columns
df.drop(columns=["Date"], inplace=True)

### **2. Handling for SARS period**
`sars_flag`: a Binary flag used to indicate whether the specific week is in SARS period or not

In [4]:
# Handle SARS peroid
sars_index = df[((df["From"] >= "2003-03-11") & (df["From"] <= "2023-06-06"))].index
df.loc[sars_index, "sars_flag"] = 1

df.fillna({"sars_flag": 0}, inplace=True)


### **3. Population**


In [5]:
path = os.getcwd()
path += "\\dataset\\Population.csv"
df_pop = pd.read_csv(path)

df_pop = df_pop.iloc[[3, 4, 6]]
df_pop.drop(columns=["Unnamed: 1"], inplace=True)
df_pop = df_pop.T
df_pop.index = range(df_pop.shape[0])
df_pop.columns = ["Year", "Reference time-point", "Population"]
df_pop.drop([0], inplace=True)
display(df_pop.head())
df_pop.info()
df_pop["Year"] = pd.to_numeric(df_pop["Year"])

df["Reference time-point"] = np.where(df["Week"] >= 23, "Year-end", "Mid-year")

df = pd.merge(df, df_pop, on=["Year", "Reference time-point"], how="left")

df.loc[df[df["Population"].isna()].index, "Population"] = df_pop.loc[df_pop[df_pop["Year"]==2026].index, "Population"].values[0]
df["Population"] = pd.to_numeric(df["Population"])

df.drop(columns="Reference time-point", inplace=True)


,Year,Reference time-point,Population
1,2002,Mid-year,6744.1
2,2002,Year-end,6725.8
3,2003,Mid-year,6730.8
4,2003,Year-end,6764.2
5,2004,Mid-year,6783.5


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 1 to 49
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Year                  49 non-null     object
 1   Reference time-point  49 non-null     object
 2   Population            49 non-null     object
dtypes: object(3)
memory usage: 1.3+ KB


### **4. School**
`school_flag`: utilized to indicate whether students need to go to school for the whole week

In [6]:
no_school_index = df[((df["Week"] >= 31) & (df["Week"] <= 35))].index
df.loc[no_school_index, "school_flag"] = 0
df.fillna({"school_flag": 1}, inplace=True)

---

In [7]:
display(df)
df.info()

,Year,Week,From,Diurnal temperature range (°C),Mean temperature (°C),Mean humidity (%),Solar radiation (MJ/m²),Air pressure (hPa),Dew point temperature (°C),Wind speed (km/h),Mean UV,H1,H3,B,AandB,ContainmentHealthIndex_Average,pandemic_flag,sars_flag,Population,school_flag
0,2002,36,2002-09-01,14.3,28.15,76.5,19.66,1008.20,23.20,7.35,4.0,1,12,3,19,0.0,0.0,0.0,6725.8,1.0
1,2002,37,2002-09-08,13.6,26.70,85.0,9.35,1006.30,23.80,10.50,3.0,0,6,4,15,0.0,0.0,0.0,6725.8,1.0
2,2002,38,2002-09-15,11.2,26.40,83.0,16.75,1011.35,23.20,13.45,4.0,0,4,4,13,0.0,0.0,0.0,6725.8,1.0
3,2002,39,2002-09-22,10.0,25.90,85.0,10.93,1012.90,23.25,13.25,3.0,0,2,6,10,0.0,0.0,0.0,6725.8,1.0
4,2002,40,2002-09-29,10.6,25.95,82.0,11.56,1012.80,22.40,10.05,3.0,0,6,2,8,0.0,0.0,0.0,6725.8,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1248,2026,31,2026-08-02,9.7,29.10,76.0,18.95,1004.50,25.00,7.00,4.0,682,438,103,1252,0.0,0.0,0.0,7518.3,0.0
1249,2026,32,2026-08-09,10.3,31.70,73.0,21.44,1000.05,26.10,9.10,5.0,713,356,89,1190,0.0,0.0,0.0,7518.3,0.0
1250,2026,33,2026-08-16,7.9,29.70,82.0,13.81,1003.10,26.00,9.50,3.0,735,245,65,1077,0.0,0.0,0.0,7518.3,0.0
1251,2026,34,2026-08-23,6.7,29.90,80.0,14.01,1001.60,26.70,8.30,3.0,846,206,60,1148,0.0,0.0,0.0,7518.3,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1253 entries, 0 to 1252
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Year                            1253 non-null   int64         
 1   Week                            1253 non-null   int64         
 2   From                            1253 non-null   datetime64[ns]
 3   Diurnal temperature range (°C)  1253 non-null   float64       
 4   Mean temperature (°C)           1253 non-null   float64       
 5   Mean humidity (%)               1253 non-null   float64       
 6   Solar radiation (MJ/m²)         1253 non-null   float64       
 7   Air pressure (hPa)              1253 non-null   float64       
 8   Dew point temperature (°C)      1253 non-null   float64       
 9   Wind speed (km/h)               1253 non-null   float64       
 10  Mean UV                         1253 non-null   float64       
 11  H1  

In [8]:
# File export 
df.to_csv("data_engineed.csv", index=False, encoding="utf-8-sig")